In [ ]:
"""
Step 1: 통합 데이터 전처리
- 8:2 Train/Val 분할 (image_id 기준)
- YOLO 포맷: symlink 이미지 + 정규화 좌표 txt 라벨
- Faster R-CNN 포맷: faster_train.csv / faster_val.csv
- data.yaml 생성

실행: python code/prep_step1.py
"""
import os
import yaml
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# ── 환경 감지 (Kaggle vs 로컬) ────────────────────────────────────────────────
import sys
import subprocess

IS_KAGGLE = os.path.exists("/kaggle/input")
if IS_KAGGLE:
    _r = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "scikit-learn", "pyyaml"],
        capture_output=True, text=True,
    )
    if _r.returncode != 0:
        print(_r.stderr)

def _find_kaggle_input(marker: str = "train.csv") -> str:
    """/kaggle/input 하위 1~2단계에서 marker 파일이 있는 디렉토리를 자동 탐색."""
    root = "/kaggle/input"
    top_dirs = sorted(os.listdir(root))
    print(f"[Kaggle] /kaggle/input 목록: {top_dirs}")

    for name in top_dirs:
        top = f"{root}/{name}"
        if not os.path.isdir(top):
            continue
        # 1단계
        if os.path.exists(f"{top}/{marker}"):
            print(f"[Kaggle] 데이터 경로 감지: {top}")
            return top
        # 2단계 (이중 폴더 구조 대응)
        for sub in sorted(os.listdir(top)):
            candidate = f"{top}/{sub}"
            if os.path.isdir(candidate) and os.path.exists(f"{candidate}/{marker}"):
                print(f"[Kaggle] 데이터 경로 감지: {candidate}")
                return candidate

    raise FileNotFoundError(
        f"'{marker}'를 포함한 디렉토리를 /kaggle/input 에서 찾을 수 없습니다.\n"
        f"Kaggle 노트북에 대회 데이터셋이 추가되어 있는지 확인하세요.\n"
        f"(우측 사이드바 → Add Data → Competition Data: amia-public-challenge-2026)"
    )

BASE_INPUT = _find_kaggle_input() if IS_KAGGLE else "amia-public-challenge-2026"
BASE_WORK  = "/kaggle/working"    if IS_KAGGLE else "."
# ─────────────────────────────────────────────────────────────────────────────

# ── 설정 ──────────────────────────────────────────────────────────────────────
TRAIN_CSV    = f"{BASE_INPUT}/train.csv"
IMG_SIZE_CSV = f"{BASE_INPUT}/img_size.csv"
IMG_DIR      = f"{BASE_INPUT}/train/train"
OUTPUT_DIR   = f"{BASE_WORK}/datasets/amiya"
VAL_RATIO    = 0.2
RANDOM_SEED  = 42
# ─────────────────────────────────────────────────────────────────────────────


def find_image(image_id: str, img_dir: str) -> tuple:
    """동적 파일 확장자 매칭으로 이미지 경로 탐색."""
    for ext in (".png", ".jpg", ".jpeg", ".dcm", ".dicom"):
        path = os.path.join(img_dir, f"{image_id}{ext}")
        if os.path.exists(path):
            return path, ext
    return None, None


def process_split(image_ids: list, split: str, grouped: dict,
                  size_lookup: dict, img_exts: dict) -> pd.DataFrame:
    records = []

    for image_id in image_ids:
        ext  = img_exts[image_id]
        src  = os.path.abspath(os.path.join(IMG_DIR, f"{image_id}{ext}"))
        sym  = os.path.abspath(f"{OUTPUT_DIR}/images/{split}/{image_id}{ext}")
        lbl  = f"{OUTPUT_DIR}/labels/{split}/{image_id}.txt"
        rows = grouped[image_id]

        # 이미지 심볼릭 링크
        if not os.path.lexists(sym):
            os.symlink(src, sym)

        # 이미지 크기
        if image_id not in size_lookup:
            print(f"  [경고] img_size 없음: {image_id}, 스킵")
            continue
        dim0 = float(size_lookup[image_id]["dim0"])  # height
        dim1 = float(size_lookup[image_id]["dim1"])  # width

        is_bg = (rows["class_name"] == "No finding").all()

        if is_bg:
            # 배경 이미지: YOLO 빈 txt, R-CNN NaN bbox
            open(lbl, "w").close()
            records.append({
                "image_id": image_id, "image_path": src,
                "class_id": np.nan,
                "x_min": np.nan, "y_min": np.nan,
                "x_max": np.nan, "y_max": np.nan,
                "dim0": dim0, "dim1": dim1,
            })
        else:
            bbox_rows = rows.dropna(subset=["x_min", "y_min", "x_max", "y_max"])
            # 유효하지 않은 bbox 제거
            valid = (
                (bbox_rows["x_max"] > bbox_rows["x_min"]) &
                (bbox_rows["y_max"] > bbox_rows["y_min"])
            )
            bbox_rows = bbox_rows[valid]

            yolo_lines = []
            for _, r in bbox_rows.iterrows():
                cid = int(r["class_id"])
                # 이미지 경계로 클리핑
                x1 = max(0.0, min(float(r["x_min"]), dim1))
                y1 = max(0.0, min(float(r["y_min"]), dim0))
                x2 = max(0.0, min(float(r["x_max"]), dim1))
                y2 = max(0.0, min(float(r["y_max"]), dim0))

                # YOLO 정규화 좌표: class cx cy w h
                cx = (x1 + x2) / 2 / dim1
                cy = (y1 + y2) / 2 / dim0
                w  = (x2 - x1) / dim1
                h  = (y2 - y1) / dim0
                yolo_lines.append(f"{cid} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")

                records.append({
                    "image_id": image_id, "image_path": src,
                    "class_id": cid,
                    "x_min": x1, "y_min": y1,
                    "x_max": x2, "y_max": y2,
                    "dim0": dim0, "dim1": dim1,
                })

            with open(lbl, "w") as f:
                f.write("\n".join(yolo_lines))

    return pd.DataFrame(records)


def main():
    # 출력 디렉토리 생성
    for split in ("train", "val"):
        os.makedirs(f"{OUTPUT_DIR}/images/{split}", exist_ok=True)
        os.makedirs(f"{OUTPUT_DIR}/labels/{split}", exist_ok=True)

    train_df    = pd.read_csv(TRAIN_CSV)
    img_size_df = pd.read_csv(IMG_SIZE_CSV)
    size_lookup = img_size_df.set_index("image_id")[["dim0", "dim1"]].to_dict("index")

    # 클래스 매핑 (class_id 0-13 → class_name)
    disease_df = train_df[train_df["class_name"] != "No finding"]
    class_map = (
        disease_df[["class_id", "class_name"]]
        .drop_duplicates()
        .sort_values("class_id")
        .reset_index(drop=True)
    )
    id2name     = dict(zip(class_map["class_id"], class_map["class_name"]))
    class_names = [id2name[i] for i in range(len(id2name))]
    assert len(class_names) == 14, f"클래스 수 오류: {len(class_names)}"

    # 유효 파일 탐색 (동적 확장자 매칭)
    all_ids   = train_df["image_id"].unique()
    valid_ids, img_exts = [], {}
    for iid in all_ids:
        path, ext = find_image(iid, IMG_DIR)
        if path:
            valid_ids.append(iid)
            img_exts[iid] = ext

    print(f"전체 image_id : {len(all_ids)}")
    print(f"유효 파일     : {len(valid_ids)}")
    if len(all_ids) - len(valid_ids) > 0:
        print(f"누락 파일     : {len(all_ids) - len(valid_ids)}")

    # 8:2 분할 (image_id 기준)
    train_ids, val_ids = train_test_split(
        valid_ids, test_size=VAL_RATIO, random_state=RANDOM_SEED
    )
    print(f"Train: {len(train_ids)} | Val: {len(val_ids)}")

    grouped = {iid: grp for iid, grp in train_df.groupby("image_id")}

    # 전처리 실행
    print("\n[Train 처리 중...]")
    train_meta = process_split(train_ids, "train", grouped, size_lookup, img_exts)
    print(f"  → {len(train_meta)} 행")

    print("[Val 처리 중...]")
    val_meta = process_split(val_ids, "val", grouped, size_lookup, img_exts)
    print(f"  → {len(val_meta)} 행")

    train_meta.to_csv(f"{OUTPUT_DIR}/faster_train.csv", index=False)
    val_meta.to_csv(  f"{OUTPUT_DIR}/faster_val.csv",   index=False)

    # data.yaml
    yaml_data = {
        "path":  os.path.abspath(OUTPUT_DIR),
        "train": "images/train",
        "val":   "images/val",
        "nc":    len(class_names),
        "names": class_names,
    }
    with open(f"{OUTPUT_DIR}/data.yaml", "w") as f:
        yaml.dump(yaml_data, f, allow_unicode=True, default_flow_style=False)

    # 배경 vs 질환 이미지 비율 통계
    train_bg = sum(1 for iid in train_ids if (grouped[iid]["class_name"] == "No finding").all())
    val_bg   = sum(1 for iid in val_ids   if (grouped[iid]["class_name"] == "No finding").all())

    print(f"\n=== 완료 ===")
    print(f"클래스 ({len(class_names)}): {class_names}")
    print(f"\nTrain 배경(정상): {train_bg} / {len(train_ids)} ({train_bg/len(train_ids)*100:.1f}%)")
    print(f"Val   배경(정상): {val_bg}   / {len(val_ids)}   ({val_bg/len(val_ids)*100:.1f}%)")
    print(f"\n출력: {OUTPUT_DIR}/")
    print(f"  data.yaml, images/{{train,val}}, labels/{{train,val}}")
    print(f"  faster_train.csv, faster_val.csv")


if __name__ == "__main__":
    main()


In [2]:
"""
Step 2: YOLO (YOLOv8m) 학습 및 평가
- Focal Loss, Mosaic/Mixup 증강
- max_det=300 (밀집 병변 대응)
- 대회 기준 mAP@IoU=0.4 산출 (torchmetrics)

실행: python code/model_yolo.py
"""
import os
import sys
import subprocess

# ── 환경 감지 및 패키지 자동 설치 (Kaggle) ───────────────────────────────────
IS_KAGGLE = os.path.exists("/kaggle/input")

if IS_KAGGLE:
    # GPU compute capability 감지: torch import 전에 nvidia-smi로 확인
    # → P100(sm_60)과 T4(sm_75)에 따라 다른 torch 버전이 필요
    _smi = subprocess.run(
        ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
        capture_output=True, text=True,
    )
    _sm = 99.0
    if _smi.returncode == 0 and _smi.stdout.strip():
        try:
            _sm = float(_smi.stdout.strip().split("\n")[0].strip())
        except ValueError:
            pass

    # Step 1: ultralytics / torchmetrics 설치 (PyPI → CPU torch 교체 발생 가능)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "ultralytics", "torchmetrics"],
                   capture_output=True, text=True)

    # Step 2: GPU SM capability에 맞는 CUDA torch 설치
    # P100 sm_60 → torch 2.3.1 (sm_60 지원 마지막 버전)
    # T4/V100+ sm_70+ → latest torch
    if _sm < 7.0:
        _torch_pkgs = ["torch==2.3.1", "torchvision==0.18.1"]
        _idx        = "https://download.pytorch.org/whl/cu121"
        print(f"[GPU] sm_{_sm} (P100) 감지 → torch 2.3.1+cu121 설치 (sm_60 마지막 지원)")
    else:
        _torch_pkgs = ["torch", "torchvision"]
        _idx        = "https://download.pytorch.org/whl/cu124"
        print(f"[GPU] sm_{_sm} 감지 → torch latest+cu124 설치")

    _r = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q"] + _torch_pkgs +
        ["--index-url", _idx],
        capture_output=True, text=True,
    )
    if _r.returncode != 0:
        print(_r.stderr)
# ─────────────────────────────────────────────────────────────────────────────

import os as _os
_os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")

import glob
import json
import yaml
import time
import torch   # pip install 완료 후 import → 올바른 버전 로드
import numpy as np
import pandas as pd
from pathlib import Path
from torchmetrics.detection import MeanAveragePrecision

def _find_kaggle_input(marker: str = "sample_submission.csv") -> str:
    """/kaggle/input 하위 1~2단계에서 marker 파일이 있는 디렉토리를 자동 탐색."""
    root = "/kaggle/input"
    top_dirs = sorted(os.listdir(root))
    print(f"[Kaggle] /kaggle/input 목록: {top_dirs}")

    for name in top_dirs:
        top = f"{root}/{name}"
        if not os.path.isdir(top):
            continue
        # 1단계
        if os.path.exists(f"{top}/{marker}"):
            print(f"[Kaggle] 데이터 경로 감지: {top}")
            return top
        # 2단계 (이중 폴더 구조 대응)
        for sub in sorted(os.listdir(top)):
            candidate = f"{top}/{sub}"
            if os.path.isdir(candidate) and os.path.exists(f"{candidate}/{marker}"):
                print(f"[Kaggle] 데이터 경로 감지: {candidate}")
                return candidate

    raise FileNotFoundError(
        f"'{marker}'를 포함한 디렉토리를 /kaggle/input 에서 찾을 수 없습니다.\n"
        f"Kaggle 노트북에 대회 데이터셋이 추가되어 있는지 확인하세요.\n"
        f"(우측 사이드바 → Add Data → Competition Data: amia-public-challenge-2026)"
    )

BASE_INPUT = _find_kaggle_input() if IS_KAGGLE else "amia-public-challenge-2026"
BASE_WORK  = "/kaggle/working"    if IS_KAGGLE else "."
# ─────────────────────────────────────────────────────────────────────────────

# ── 설정 ──────────────────────────────────────────────────────────────────────
TEST_DIR           = f"{BASE_INPUT}/test/test"
SAMPLE_SUBMISSION  = f"{BASE_INPUT}/sample_submission.csv"
SUBMISSION_CSV     = f"{BASE_WORK}/submission.csv"
KAGGLE_COMPETITION = "amia-public-challenge-2026"
SUBMIT_MESSAGE     = "YOLOv8m baseline"
INFER_CONF         = 0.01            # PR Curve 컷오프 방지: 낮게 설정해야 Kaggle mAP 정확히 산출
IMG_SIZE_CSV       = f"{BASE_INPUT}/img_size.csv"
NO_FINDING_THRESH  = 0.15            # 최고 신뢰도가 이 이하면 No Finding 강제 할당
                                     # (INFER_CONF 낮춰 잡음 박스가 많아질 때 정상 점수 보존)

DATA_YAML   = f"{BASE_WORK}/datasets/amiya/data.yaml"
MODEL_NAME  = "yolov8m.pt"   # yolo11m.pt 로 변경 시 최신 YOLO11 사용
EPOCHS      = 100
IMGSZ       = 1024   # 640→1024: 소형 병변 탐지력 극대화 (16GB VRAM 활용)
BATCH       = 28     # GPU당 14장 @ imgsz=1024 → ~11.5GB 목표
                     # 32(16/GPU)는 14.5~14.7GB로 여유 없음 → OOM 위험
MAX_DET     = 300
CACHE       = False  # RAM(26GB)/Disk(19GB) 모두 30GB 필요로 캐시 불가 → 비활성화
# fl_gamma: ultralytics 8.x에서 공개 파라미터 제거됨.
# YOLOv8 내부적으로 Focal Loss(gamma≈1.5)를 고정 적용함.
# 클래스 불균형 대응은 cls_pw(BCE positive weight)로 대체.
CLS_PW      = 1.0            # 소수 클래스 가중치 (>1.0이면 양성 페널티 강화)
CUTMIX      = 0.1            # 8.4.56에서 cutmix 공식 지원
MOSAIC      = 1.0
MIXUP       = 0.1
WORKERS     = 2 if IS_KAGGLE else 4   # Kaggle CPU 코어 수 제한 반영
PROJECT     = f"{BASE_WORK}/runs/yolo"
NAME        = "amiya_yolov8m"
INFER_BATCH = 16             # mAP@0.4 계산 시 배치 크기
RESULT_JSON = f"{BASE_WORK}/runs/yolo/yolo_map40.json"
# ─────────────────────────────────────────────────────────────────────────────


def load_yolo_gt(lbl_path: str, img_w: int, img_h: int) -> tuple:
    """YOLO 정규화 좌표 → xyxy 픽셀 좌표 변환."""
    boxes, labels = [], []
    if not os.path.exists(lbl_path):
        return torch.zeros((0, 4)), torch.zeros((0,), dtype=torch.long)
    with open(lbl_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            cid, cx, cy, w, h = int(parts[0]), float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
            x1 = (cx - w / 2) * img_w
            y1 = (cy - h / 2) * img_h
            x2 = (cx + w / 2) * img_w
            y2 = (cy + h / 2) * img_h
            boxes.append([x1, y1, x2, y2])
            labels.append(cid)
    if not boxes:
        return torch.zeros((0, 4)), torch.zeros((0,), dtype=torch.long)
    return torch.tensor(boxes, dtype=torch.float32), torch.tensor(labels, dtype=torch.long)


def compute_map40(model, val_img_dir: str, val_lbl_dir: str) -> dict:
    """val set 전체 배치 추론 후 mAP@IoU=0.4 산출."""
    metric     = MeanAveragePrecision(iou_thresholds=[0.4], class_metrics=True)
    img_paths  = sorted(glob.glob(os.path.join(val_img_dir, "*")))
    total      = len(img_paths)

    for start in range(0, total, INFER_BATCH):
        batch = img_paths[start:start + INFER_BATCH]
        results = model.predict(batch, verbose=False, max_det=MAX_DET, conf=0.001)

        preds, gts = [], []
        for img_path, r in zip(batch, results):
            img_h, img_w = r.orig_shape[:2]
            stem     = Path(img_path).stem
            lbl_path = os.path.join(val_lbl_dir, f"{stem}.txt")

            # 예측
            if r.boxes is not None and len(r.boxes):
                p_boxes  = r.boxes.xyxy.cpu()
                p_scores = r.boxes.conf.cpu()
                p_labels = r.boxes.cls.cpu().long()
            else:
                p_boxes  = torch.zeros((0, 4))
                p_scores = torch.zeros((0,))
                p_labels = torch.zeros((0,), dtype=torch.long)

            # 정답
            gt_boxes, gt_labels = load_yolo_gt(lbl_path, img_w, img_h)

            preds.append({"boxes": p_boxes, "scores": p_scores, "labels": p_labels})
            gts.append(  {"boxes": gt_boxes, "labels": gt_labels})

        metric.update(preds, gts)
        done = min(start + INFER_BATCH, total)
        print(f"  mAP@0.4 평가 진행: {done}/{total}", end="\r")

    print()
    return metric.compute()


def generate_submission(model, output_path: str = SUBMISSION_CSV) -> str:
    """Test set 추론 → PredictionString 포맷 submission.csv 생성."""
    img_paths = sorted(glob.glob(os.path.join(TEST_DIR, "*")))
    total = len(img_paths)

    # 안전장치: 테스트 이미지가 없으면 즉시 실패
    if total == 0:
        raise FileNotFoundError(
            f"TEST_DIR에 이미지가 없습니다: {TEST_DIR}\n"
            f"경로와 데이터셋 마운트 상태를 확인하세요."
        )

    # img_size.csv 로드: r.orig_shape → 실제 원본 해상도 스케일링 용도
    img_size_lookup: dict = {}
    if os.path.exists(IMG_SIZE_CSV):
        _df = pd.read_csv(IMG_SIZE_CSV).set_index("image_id")
        img_size_lookup = _df[["dim0", "dim1"]].to_dict("index")

    predictions = {}

    for start in range(0, total, INFER_BATCH):
        batch = img_paths[start:start + INFER_BATCH]
        results = model.predict(batch, verbose=False, max_det=MAX_DET, conf=INFER_CONF)

        for img_path, r in zip(batch, results):
            image_id = Path(img_path).stem
            orig_h, orig_w = r.orig_shape[:2]

            # 원본 해상도 복원: r.orig_shape와 img_size.csv가 다를 경우 스케일링
            if image_id in img_size_lookup:
                true_h = img_size_lookup[image_id]["dim0"]
                true_w = img_size_lookup[image_id]["dim1"]
            else:
                true_h, true_w = orig_h, orig_w  # fallback: 스케일링 없음
            scale_x = true_w / orig_w
            scale_y = true_h / orig_h

            if r.boxes is not None and len(r.boxes):
                max_conf = float(r.boxes.conf.max().item())

                # 정상 클래스 보존: 최고 신뢰도가 임계값 미만 → No Finding 강제 할당
                if max_conf < NO_FINDING_THRESH:
                    predictions[image_id] = "14 1.0 0 0 1 1"
                else:
                    parts = []
                    for box, score, cls in zip(
                        r.boxes.xyxy.cpu().numpy(),
                        r.boxes.conf.cpu().numpy(),
                        r.boxes.cls.cpu().numpy(),
                    ):
                        # 원본 해상도로 스케일링 후 float 그대로 제출
                        x1 = box[0] * scale_x
                        y1 = box[1] * scale_y
                        x2 = box[2] * scale_x
                        y2 = box[3] * scale_y
                        parts.append(f"{int(cls)} {score:.4f} {x1:.1f} {y1:.1f} {x2:.1f} {y2:.1f}")
                    predictions[image_id] = " ".join(parts)
            else:
                predictions[image_id] = "14 1.0 0 0 1 1"

        done = min(start + INFER_BATCH, total)
        print(f"  Test 추론: {done}/{total}", end="\r")

    print()
    sample_df = pd.read_csv(SAMPLE_SUBMISSION)
    sample_df["PredictionString"] = sample_df["image_id"].map(
        lambda x: predictions.get(x, "14 1.0 0 0 1 1")
    )
    sample_df.to_csv(output_path, index=False)
    print(f"Submission 저장: {output_path} ({len(sample_df)}행)")
    return output_path


def submit_to_kaggle(submission_path: str) -> float:
    """Kaggle CLI로 제출하고 업로드 소요 시간을 반환."""
    cmd = [
        "kaggle", "competitions", "submit",
        "-c", KAGGLE_COMPETITION,
        "-f", submission_path,
        "-m", SUBMIT_MESSAGE,
    ]
    print(f"\n[Kaggle 제출] {' '.join(cmd)}")
    t0 = time.time()
    result = subprocess.run(cmd, capture_output=True, text=True)
    elapsed = time.time() - t0

    print(f"업로드 소요 시간: {elapsed:.1f}초")
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.returncode != 0 and result.stderr.strip():
        print(f"[오류] {result.stderr.strip()}")
    return elapsed


def _patch_ultralytics_ddp():
    """
    ultralytics trainer.py의 DDP 초기화에 find_unused_parameters=True 추가.

    문제: YOLOv8 DFL 파라미터는 background-only 배치에서 gradient를 받지 못함.
    DDP 기본값(find_unused_parameters=False)은 이를 허용하지 않아 RuntimeError 발생.
    해결: 설치된 소스를 직접 수정해 DDP 서브프로세스에도 패치가 적용되도록 함.
    """
    import re
    from pathlib import Path

    try:
        import ultralytics
        trainer_path = Path(ultralytics.__file__).parent / "engine" / "trainer.py"
    except Exception as e:
        print(f"[DDP 패치] ultralytics 경로 탐색 실패: {e}")
        return

    src = trainer_path.read_text()

    if "find_unused_parameters" in src:
        print("[DDP 패치] 이미 find_unused_parameters 적용됨")
        return

    lines = src.splitlines()

    # DistributedDataParallel 초기화 시작 라인 탐색
    ddp_start = next(
        (i for i, line in enumerate(lines)
         if "nn.parallel.DistributedDataParallel(" in line or "DDP(self.model" in line),
        None,
    )
    if ddp_start is None:
        print("[DDP 패치] DistributedDataParallel 라인 미발견")
        return

    # 실제 코드 블록 출력 (디버깅 - 정확한 포맷 확인)
    block_end = min(ddp_start + 10, len(lines))
    print(f"[DDP 패치] L{ddp_start+1}~L{block_end} 코드 블록:")
    for i in range(ddp_start, block_end):
        print(f"  L{i+1}: {lines[i]}")

    # device_ids=[RANK] 라인 탐색 (DDP 블록 내 10줄 이내)
    device_line = next(
        (i for i in range(ddp_start, min(ddp_start + 10, len(lines)))
         if "device_ids=[RANK]" in lines[i]),
        None,
    )
    if device_line is None:
        print("[DDP 패치] device_ids=[RANK] 라인 미발견")
        return

    # device_ids 라인의 들여쓰기 수준 유지
    indent = " " * (len(lines[device_line]) - len(lines[device_line].lstrip()))

    # trailing comma 확보
    if not lines[device_line].rstrip().endswith(","):
        lines[device_line] = lines[device_line].rstrip() + ","

    # device_ids 다음 줄에 find_unused_parameters=True 삽입
    lines.insert(device_line + 1, f"{indent}find_unused_parameters=True,")

    trainer_path.write_text("\n".join(lines) + ("\n" if src.endswith("\n") else ""))
    print(f"[DDP 패치 완료] L{device_line+1} 뒤에 find_unused_parameters=True 삽입")


def to_serializable(obj):
    if isinstance(obj, torch.Tensor):
        return obj.tolist()
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.floating, np.integer)):
        return obj.item()
    return obj


def main():
    from ultralytics import YOLO

    # 디바이스 선택 (Dual GPU → DDP)
    if torch.cuda.is_available():
        n_gpu  = torch.cuda.device_count()
        device = [0, 1] if n_gpu >= 2 else 0
        print(f"디바이스: GPU x{n_gpu}{' (DDP)' if n_gpu >= 2 else ''}")
    elif torch.backends.mps.is_available():
        device = "mps"
        print("디바이스: MPS")
    else:
        device = "cpu"
        print("디바이스: CPU")

    # DDP 사용 시 background-only 배치에서 DFL 파라미터가 gradient를 받지 못하는 문제 해결.
    # ultralytics가 DDP를 temp 스크립트 서브프로세스로 실행하므로
    # 설치된 소스를 직접 패치해야 서브프로세스에도 적용됨.
    if isinstance(device, list):
        _patch_ultralytics_ddp()

    with open(DATA_YAML) as f:
        data_cfg = yaml.safe_load(f)
    val_img_dir = os.path.join(data_cfg["path"], "images", "val")
    val_lbl_dir = os.path.join(data_cfg["path"], "labels", "val")

    # ── 학습 ──────────────────────────────────────────────────────────────────
    model = YOLO(MODEL_NAME)
    model.train(
        data      = DATA_YAML,
        epochs    = EPOCHS,
        imgsz     = IMGSZ,
        batch     = BATCH,
        device    = device,
        workers   = WORKERS,
        max_det   = MAX_DET,
        cls_pw    = CLS_PW,
        cutmix    = CUTMIX,
        mosaic    = MOSAIC,
        mixup     = MIXUP,
        cache     = CACHE,
        project   = PROJECT,
        name      = NAME,
        exist_ok  = True,
        verbose   = True,
    )

    # ── 최선 가중치로 mAP@0.4 평가 ──────────────────────────────────────────
    best_weights = os.path.join(PROJECT, NAME, "weights", "best.pt")
    if not os.path.exists(best_weights):
        best_weights = os.path.join(PROJECT, NAME, "weights", "last.pt")
    print(f"\n가중치 로드: {best_weights}")
    best_model = YOLO(best_weights)

    print("mAP@0.4 계산 중...")
    map40_result = compute_map40(best_model, val_img_dir, val_lbl_dir)

    map40 = map40_result.get("map")
    if map40 is not None:
        map40_val = map40.item() if isinstance(map40, torch.Tensor) else map40
        print(f"\nmAP@0.4: {map40_val:.4f}")
    else:
        map40_val = 0.0
        print("mAP@0.4: 계산 실패")

    # ── 결과 저장 ─────────────────────────────────────────────────────────────
    serializable = {k: to_serializable(v) for k, v in map40_result.items()}
    serializable.update({
        "model":    MODEL_NAME,
        "map40":    map40_val,
        "epochs":   EPOCHS,
        "imgsz":    IMGSZ,
        "cls_pw":   CLS_PW,
        "cutmix":   CUTMIX,
        "max_det":  MAX_DET,
    })
    os.makedirs(os.path.dirname(RESULT_JSON), exist_ok=True)
    with open(RESULT_JSON, "w") as f:
        json.dump(serializable, f, indent=2, ensure_ascii=False)
    print(f"결과 저장: {RESULT_JSON}")

    # ── Test 추론 → Submission → Kaggle 제출 ─────────────────────────────────
    print("\nTest set 추론 및 submission 생성 중...")
    submission_path = generate_submission(best_model)

    elapsed = submit_to_kaggle(submission_path)
    serializable["kaggle_upload_seconds"] = round(elapsed, 1)
    with open(RESULT_JSON, "w") as f:
        json.dump(serializable, f, indent=2, ensure_ascii=False)


if __name__ == "__main__":
    main()


[GPU] sm_7.5 감지 → torch latest+cu124 설치
[Kaggle] /kaggle/input 목록: ['competitions']
[Kaggle] 데이터 경로 감지: /kaggle/input/competitions/amia-public-challenge-2026
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
디바이스: GPU x2 (DDP)
[DDP 패치] 이미 find_unused_parameters 적용됨
Ultralytics 8.4.67 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
                                                       CUDA:1 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=28, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=1.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.1, data=/kaggle/wor

/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Encountered more than 100 detections in a single image. This means that certain detections with the lowest scores will be ignored, that may have an undesirable impact on performance. Please consider adjusting the `max_detection_threshold` to suit your use case. To disable this warning, set attribute class `warn_on_many_detections=False`, after initializing the metric.
  warnings.warn(*args, **kwargs)


  mAP@0.4 평가 진행: 1715/1715

mAP@0.4: 0.3913
결과 저장: /kaggle/working/runs/yolo/yolo_map40.json

Test set 추론 및 submission 생성 중...
  Test 추론: 6427/6427
Submission 저장: /kaggle/working/submission.csv (6427행)

[Kaggle 제출] kaggle competitions submit -c amia-public-challenge-2026 -f /kaggle/working/submission.csv -m YOLOv8m baseline
업로드 소요 시간: 4.0초
Successfully submitted to AMIA Public Challenge 2026
